# SecureCode AI: аудит демонстрационного уязвимого проекта

Notebook проходит по всей цепочке из задания на небольшом уязвимом проекте:

1. **Загрузка кода** (Python/JS/Go) из Git-снимка и выбор инструментов.
2. **Инструменты (Tools)**: AST и таблица символов, поиск захардкоженных секретов,
   уязвимые версии библиотек (OSV), поиск CWE-паттернов.
3. **Единый контракт**: находки как `Evidence` / `DiscoveryCandidate` с CWE и строками.
4. **Аудитор и Архитектор** на LLM: независимый поиск, сверка с детерминированным
   lane, Auto-Fix patch, проверка патча в эфемерной копии.
5. **Отчёт** `.md`/`.html` с классификацией OWASP Top 10.
6. **Метрики** бенчмарка на 600 кейсах CVEfixes.

Модель выбирается переменной `SECURECODE_DEMO_PROVIDER`: `local` (Ollama + Qwen,
по умолчанию), `deepseek` (ключ `DEEPSEEK_API_KEY` или строка в `.env`) или
`offline` (без модели). Запрос к OSV отправляет только имя и версию пакета;
`SECURECODE_DEMO_OSV=0` отключает его.

In [1]:
import json
import os
from pathlib import Path

from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
PROVIDER = os.environ.get("SECURECODE_DEMO_PROVIDER", "local")
USE_OSV = os.environ.get("SECURECODE_DEMO_OSV", "1") != "0"
print("repository:", ROOT.name, "| provider:", PROVIDER, "| OSV:", USE_OSV)

repository: SecureCode AI | provider: local | OSV: True


## 1. Демонстрационный проект

Три файла на трёх языках и манифест зависимостей. В `app.py` SQL-инъекция через
конкатенацию и захардкоженный пароль; `requirements.txt` фиксирует старую версию
`requests` с известными уязвимостями.

In [2]:
PROJECT = {
    "app.py": (
        "import sqlite3\n\n"
        'DB_PASSWORD = "correct-horse-battery-staple"\n\n\n'
        "def find_user(conn, name):\n"
        "    return conn.execute(\"SELECT * FROM users WHERE name = '\" + name + \"'\")\n"
    ),
    "server.js": (
        "const express = require('express');\n"
        "const app = express();\n"
        "app.get('/user', (req, res) => {\n"
        "  db.query('SELECT * FROM users WHERE id = ' + req.query.id, (err, rows) => res.json(rows));\n"
        "});\n"
    ),
    "main.go": (
        "package main\n\n"
        'import "fmt"\n\n'
        "func main() {\n"
        '    fmt.Println("hello")\n'
        "}\n"
    ),
    "requirements.txt": "requests==2.19.0\n",
}
for name, text in PROJECT.items():
    print(f"--- {name}\n{text}")

--- app.py
import sqlite3

DB_PASSWORD = "correct-horse-battery-staple"


def find_user(conn, name):
    return conn.execute("SELECT * FROM users WHERE name = '" + name + "'")

--- server.js
const express = require('express');
const app = express();
app.get('/user', (req, res) => {
  db.query('SELECT * FROM users WHERE id = ' + req.query.id, (err, rows) => res.json(rows));
});

--- main.go
package main

import "fmt"

func main() {
    fmt.Println("hello")
}

--- requirements.txt
requests==2.19.0



## 2. Загрузчик и инструменты

`execute_deterministic_children` читает файлы из неизменяемого Git-снимка (здесь
объекты Git собираются в памяти), определяет язык каждого файла и запускает
нужные инструменты в изолированном дочернем процессе.

In [3]:
import hashlib

from securecode_ai.adapters.dependency_scanning_osv import BoundedOsvScanner
from securecode_ai.adapters.product_execution import execute_deterministic_children
from securecode_ai.adapters.secret_detection import SecretFingerprintKey


class GitObjects:
    """Minimal in-memory Git object store (blob/tree/commit) for the demo snapshot."""

    def __init__(self):
        self.contents = {}

    def add(self, kind, content):
        header = f"{kind} {len(content)}\0".encode()
        oid = hashlib.sha1(header + content, usedforsecurity=False).hexdigest()
        self.contents[kind, oid] = content
        return oid

    def read(self, kind, oid, *, max_bytes):
        return self.contents[kind, oid]


objects = GitObjects()
tree = objects.add(
    "tree",
    b"".join(
        b"100644 " + name.encode() + b"\0" + bytes.fromhex(objects.add("blob", text.encode()))
        for name, text in sorted(PROJECT.items())
    ),
)
head = objects.add("commit", f"tree {tree}\n\ndemo".encode())

execution = execute_deterministic_children(
    reader=objects,
    head_sha=head,
    tenant_id="demo",
    repository_id="demo/vulnerable-app",
    content_key=b"c" * 32,
    fingerprint_key=SecretFingerprintKey("demo", b"k" * 32),
    scanner=BoundedOsvScanner() if USE_OSV else None,
)
print("selected tools:", ", ".join(execution.selected_child_ids))
print("complete:", execution.is_complete, "| obstacles:", execution.obstacles or "none")

selected tools: python_parse_symbols, secret_scan, dependency_scan, cwe89_scan
complete: True | obstacles: none


### Поиск захардкоженных секретов

Значение секрета никогда не выводится: только вид, место и отпечаток.

In [4]:
for result in execution.secrets.results:
    for candidate in result.candidates:
        row = candidate.location.start_point.row + 1
        print(f"{candidate.path}:{row}  {candidate.kind.value}  {candidate.redaction}")

app.py:3  assigned_credential  <redacted:assigned_credential>


### Уязвимые версии библиотек (OSV)

In [5]:
if execution.dependencies is None:
    print("dependency scan skipped")
else:
    for result in execution.dependencies.results:
        for coordinate in result.coordinates:
            ids = [a.advisory_id for a in result.advisories if a.coordinate == coordinate]
            print(f"{coordinate.manifest_path}: {coordinate.name}=={coordinate.version}"
                  f" -> {len(ids)} advisories: {', '.join(ids[:5])}{' ...' if len(ids) > 5 else ''}")

requirements.txt: requests==2.19.0 -> 10 advisories: GHSA-9hjg-9r4m-mvj7, GHSA-9wx4-h78v-vm56, GHSA-gc5v-m9x4-r6x2, GHSA-j8r2-6x86-q33q, GHSA-x84v-xcm2-53pg ...


## 3. Единый контракт

Каждая находка инструмента нормализуется в `Evidence` (файл, строки, SHA-256
содержимого, производитель, класс данных) и `DiscoveryCandidate` (CWE, lane
происхождения). Исходный код в контракт не копируется.

In [6]:
import re

graph = execution.scan.graph
signals = [item for item in graph.evidence if item.evidence_kind.value == "scanner_signal"]
print(f"{len(graph.evidence)} evidence records, {len(signals)} scanner signals:")
for evidence in signals:
    location = evidence.location
    rule = re.sub(r"-[0-9a-f]{64}$", "", evidence.evidence_id)
    cwe = re.search(r"cwe-?(\d+)", rule, re.IGNORECASE)
    print(json.dumps({
        "rule": rule,
        "cwe_id": f"CWE-{cwe.group(1)}" if cwe else {"product-sql": "CWE-89"}.get(rule),
        "evidence_kind": evidence.evidence_kind.value,
        "path": location.path,
        "lines": [location.start.line, location.end.line],
        "content_sha256": location.content_sha256[:16] + "...",
        "producer": evidence.producer.producer_id,
        "data_class": evidence.data_class.value,
    }, indent=2))
for candidate in graph.candidates:
    print("candidate", candidate.candidate_id[:24], "| origin:", candidate.candidate_origin.value,
          "| evidence:", len(candidate.evidence_ids))

7 evidence records, 3 scanner signals:
{
  "rule": "product-cwe-306",
  "cwe_id": "CWE-306",
  "evidence_kind": "scanner_signal",
  "path": "server.js",
  "lines": [
    3,
    5
  ],
  "content_sha256": "0304e8c047e185d9...",
  "producer": "securecode-first-party-static",
  "data_class": "DC3_CONFIDENTIAL_SOURCE"
}
{
  "rule": "product-cwe-798",
  "cwe_id": "CWE-798",
  "evidence_kind": "scanner_signal",
  "path": "app.py",
  "lines": [
    3,
    3
  ],
  "content_sha256": "2166cb5c1d7ce2bc...",
  "producer": "securecode-first-party-static",
  "data_class": "DC3_CONFIDENTIAL_SOURCE"
}
{
  "rule": "product-sql",
  "cwe_id": "CWE-89",
  "evidence_kind": "scanner_signal",
  "path": "server.js",
  "lines": [
    4,
    4
  ],
  "content_sha256": "0304e8c047e185d9...",
  "producer": "securecode-first-party-static",
  "data_class": "DC3_CONFIDENTIAL_SOURCE"
}
candidate candidate-033100915f8449 | origin: deterministic | evidence: 1
candidate candidate-57daa4358da7ac | origin: deterministic 

## 4. Аудитор и Архитектор

Демо P9.17 на фикстуре с SQL-инъекцией: Аудитор (LLM) независимо ищет уязвимости,
результат сверяется с детерминированным сканером; только при совпадении Архитектор
предлагает patch. Патч применяется во временной копии, файл парсится и повторно
сканируется. Исходный checkout не меняется.

In [7]:
import subprocess
import sys
import tempfile

output = Path(tempfile.mkdtemp(prefix="securecode-notebook-")) / "report"
completed = subprocess.run(
    [sys.executable, "-I", "demo/p917_real_local_demo.py",
     "--repository", "demo/fixtures/real-local-cwe89",
     "--output", str(output), "--provider", PROVIDER],
    capture_output=True, text=True, check=False,
)
manifest = json.loads(completed.stdout)
print("outcome:", manifest["outcome"])
print("model:", manifest.get("provider_runtime", {}).get("model_id"))

outcome: COMPLETED
model: qwen2.5-coder:7b-instruct-q4_K_M


## 5. Отчёт с классификацией OWASP Top 10

In [8]:
display(Markdown((output / "security-report.md").read_text(encoding="utf-8")))

# SecureCode AI security report

Snapshot `887916cf80e9417a`, 1 Python file(s).

| | |
| --- | --- |
| Outcome | COMPLETED |
| Model | qwen2.5-coder:7b-instruct-q4_K_M via authorized_loopback |
| Auditor (discovery) | SUCCEEDED |
| Lane agreement | AGREED |
| Architect (repair) | SUCCEEDED |
| Patch | PROPOSED |
| Validation | PASSED (parse PARSED, rescan signals 0) |
| Source checkout changed | no |

## Findings (OWASP Top 10)

| CWE | OWASP Top 10 | Severity | Location | Found by |
| --- | --- | --- | --- | --- |
| CWE-89 | A03:2021 Injection | HIGH | `app.py:5` | deterministic + model |

## Auto-Fix

Proposed by the Architect model and validated in an ephemeral copy (not applied to the source checkout):

```diff
--- a/app.py
+++ b/app.py
@@ -2,4 +2,4 @@
 
 
 def find_user(conn, name):
-    return conn.execute("SELECT * FROM users WHERE name = '" + name + "'")
+    return conn.execute('SELECT * FROM users WHERE name = ?', (name,))
```

## Limitations

- One bounded Python CWE-89 rule is demonstrated; this is not a general security verdict.
- The raw model response is not retained; reports keep metadata and the validated patch only.
- Static parse and rescan do not execute the repository and do not prove behavioral correctness.
- A provider, schema, identity, or validation non-success is INDETERMINATE rather than clean or PASS.
- Lane agreement is only a diagnostic comparison, not Auditor/Skeptic confirmation or proof of accuracy.


## 6. Метрики на 600 кейсах CVEfixes

Сохранённые результаты бенчмарка (`report/submission-benchmark/aggregate.json`).
Hybrid объединяет детерминированный lane и DeepSeek; на held-out части он на уровне
Semgrep, превосходство не доказано. Подробности и ограничения в
[отчёте бенчмарка](../report/submission-benchmark/README.md).

In [9]:
aggregate = json.loads(Path("report/submission-benchmark/aggregate.json").read_text())
rows = ["| Lane | Precision | Recall | F1 | Completed |", "| --- | ---: | ---: | ---: | ---: |"]
for lane, value in aggregate["lanes"].items():
    rows.append(
        f"| {lane} | {value['precision']:.1%} | {value['recall']:.1%} | {value['f1']:.1%}"
        f" | {value['completion_rate']:.1%} |"
    )
display(Markdown("\n".join(rows)))

| Lane | Precision | Recall | F1 | Completed |
| --- | ---: | ---: | ---: | ---: |
| deterministic_only | 49.0% | 25.7% | 33.7% | 84.7% |
| full_hybrid | 49.9% | 35.8% | 41.7% | 84.7% |
| model_native | 50.5% | 16.8% | 25.2% | 100.0% |
| one_shot | 52.0% | 22.7% | 31.6% | 100.0% |
| scanner_seeded | 48.8% | 6.7% | 11.7% | 84.7% |
| semgrep | 50.0% | 34.3% | 40.7% | 100.0% |